# Claim types, subkinds and argument structure (taxonomy v0.10)

**Goal:** test the v0.10 taxonomy on the same 4 samples as v0.9, with haiku and opus.

| Claim type | Colour | Subkinds |
|---|---|---|
| RECOMMENDATION | red | `trade_idea` or `rating` |
| PREDICTION | yellow | `event`, `directional`, `quantitative` (can combine) |
| CAUSAL | purple | one topic: MS / VAL / POS / SENT / EVT |
| VIEW | blue | one topic |
| OBSERVATION | grey | `datapoint` or `estimate`, only if it states a number |
| SUPPOSITION | blue-grey | one topic; modality always HYPOTHETICAL |

What is new in v0.10:
- The model lists the arguments first and then builds each one out.
- New roles: `concessions`, `implications` and `alternatives`.
- Scenarios are separate structures, referenced from argument roles.
- Recommendations, predictions, and valuation and positioning views are on a must-keep list.

**Pipeline** (repo root):

```
.venv\Scripts\python.exe scripts\run_samples.py --limit 4
```

**Questions**

1. Are runs valid, and how do cost and structure compare with v0.9 and v0.8?
2. **Hierarchy:** are arguments linked, and are recommendations linked to the views that motivate them?
3. **Coverage:** are must-keep claims still excluded, and which v0.9 claims are missing?
4. Do haiku and opus agree on type and subkind?
5. Where did the v0.9 function·kind pairs go, especially ASSESSMENT·CAUSE?
6. How are suppositions, conditions and scenarios used?
7. Are CAUSAL claims mechanisms rather than predictions?
8. How is attribution used?

The probes in this notebook are regex heuristics, not extractions.

In [ ]:
import json
import re
import sys
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from incolor.argumentation.evaluate import match_statements
from incolor.argumentation.validate import argument_edges, scenario_edges
from incolor.argumentation.writer import load_argument_map

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
RUNS_DIR = ROOT / "outputs" / "samples_v10"
DATA_DIR = RUNS_DIR / "claims_analysis"
V09_DIR = ROOT / "outputs" / "samples_v09"
V08_DIR = ROOT / "outputs" / "samples_v08"
TYPES = ["RECOMMENDATION", "PREDICTION", "CAUSAL", "VIEW", "OBSERVATION", "SUPPOSITION"]
TOPICS = ["market_state", "valuation", "positioning", "sentiment", "event"]
SUBKINDS = {
    "RECOMMENDATION": ["trade_idea", "rating"],
    "PREDICTION": ["event", "directional", "quantitative"],
    "OBSERVATION": ["datapoint", "estimate"],
    "VIEW": TOPICS,
    "CAUSAL": TOPICS,
    "SUPPOSITION": TOPICS,
}
ROLES = [
    "core_claim", "supporting", "rationale", "risks", "conditions", "catalysts", "qualifiers", "concessions",
    "implications", "alternatives",
]
MODELS = ["haiku", "opus"]
PIVOT_MODEL = "opus"
JSON_COLUMNS = ["evidence_ids", "evidence_texts", "roles", "argument_ids"]
CACHES = ["pairwise.csv", "v09_alignment.csv"]

# Rebuild the CSV when the set of valid maps has changed; the alignment caches depend on it.
claims_path = DATA_DIR / "claims_all.csv"
valid_maps = {p.relative_to(ROOT).as_posix() for p in RUNS_DIR.glob("*/*/argument_map.yaml")}
built_maps = (
    set(pd.read_csv(claims_path, usecols=["map_path"], encoding="utf-8-sig")["map_path"])
    if claims_path.exists()
    else set()
)
if built_maps != valid_maps:
    sys.path.insert(0, str(ROOT / "scripts"))
    import build_claims_csv

    build_claims_csv.main([
        "--runs-root", str(RUNS_DIR), "--texts-dir", str(RUNS_DIR / "inputs"), "--no-gold",
        "--pivot-model", PIVOT_MODEL, "--out", str(DATA_DIR),
    ])
    for name in CACHES:
        (DATA_DIR / name).unlink(missing_ok=True)

claims = pd.read_csv(claims_path, encoding="utf-8-sig")
for col in JSON_COLUMNS:
    claims[col] = claims[col].map(lambda v: json.loads(v) if isinstance(v, str) and v else [])
claims["subkind_list"] = claims["subkinds"].map(lambda v: v.split("|") if isinstance(v, str) and v else [])
claims["sub"] = claims["subkind_list"].map("+".join)
DOCS = sorted({p.parent.parent.name for p in RUNS_DIR.glob("*/*/summary.json")})
LABEL = {doc: doc.split("_-_")[0].replace("__", " ").replace("_", " ")[:24] for doc in DOCS}
claims["label"] = claims["doc"].map(LABEL)

pd.set_option("display.max_colwidth", 160)
pd.set_option("display.max_rows", 200)
print(f"{len(claims)} statements from {claims['run_id'].nunique()} valid runs over {claims['doc'].nunique()} samples")
pd.crosstab(claims["label"], claims["model"]).reindex(columns=MODELS, fill_value=0)

In [ ]:
SHOW = ["label", "model", "statement_id", "claim_type", "sub", "modality", "attribution", "statement"]


def probe(df, patterns, col="statement"):
    """One boolean column per named regex (case-insensitive) matched against ``col``."""
    return pd.DataFrame(
        {name: df[col].str.contains(p, case=False, regex=True, na=False) for name, p in patterns.items()},
        index=df.index,
    )


def heatmap(table, title, xlabel, ylabel, ax=None):
    """Row-normalised heat map of a count table, annotated with the counts."""
    if ax is None:
        _, ax = plt.subplots(figsize=(1.5 + 0.9 * table.shape[1], 1 + 0.55 * table.shape[0]))
    shares = table.div(table.sum(axis=1).replace(0, 1), axis=0)
    ax.imshow(shares, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(table.shape[1]), table.columns, rotation=45, ha="right")
    ax.set_yticks(range(table.shape[0]), table.index)
    for i in range(table.shape[0]):
        for j in range(table.shape[1]):
            if table.iat[i, j]:
                ax.text(j, i, table.iat[i, j], ha="center", va="center", fontsize=8)
    ax.set(title=title, xlabel=xlabel, ylabel=ylabel)
    return ax


def load_maps(runs_dir):
    """``{(doc, model): ArgumentMap}`` for every valid map under ``runs_dir``."""
    return {(p.parent.parent.name, p.parent.name): load_argument_map(p) for p in sorted(runs_dir.glob("*/*/argument_map.yaml"))}


maps10 = load_maps(RUNS_DIR)
maps09 = {key: m for key, m in load_maps(V09_DIR).items() if key[0] in DOCS}

# One row per (statement, argument, role).
memberships = claims[["run_id", "label", "model", "statement_id", "claim_type", "sub", "roles"]].explode("roles")
memberships = memberships.dropna(subset=["roles"]).reset_index(drop=True)
memberships["role"] = memberships["roles"].map(lambda r: r["role"])
memberships = memberships.drop(columns="roles")

## 1. Runs: validity, cost and structure, against v0.9 and v0.8

In [ ]:
def load_runs(runs_dir):
    runs = pd.DataFrame([json.loads(p.read_text(encoding="utf-8")) for p in sorted(runs_dir.glob("*/*/summary.json"))])
    runs = runs[runs["doc"].isin(DOCS)].copy()
    runs["label"] = runs["doc"].map(LABEL)
    runs["stmts_per_1k"] = runs["statements"] / runs["src_chars"] * 1000
    return runs


def summarise(runs):
    """Per-model totals and medians over the run summaries; columns missing in older batches are skipped."""
    medians = [c for c in ["seconds", "statements", "stmts_per_1k", "arguments", "edges", "isolated", "max_chain",
                           "exclusions", "scenarios", "scenario_links"] if c in runs]
    out = runs.groupby("model").agg(runs=("doc", "size"), valid=("ok", "sum"), cost_usd=("cost_usd", "sum"),
                                    mean_attempts=("attempts", "mean"))
    return out.join(runs.groupby("model")[medians].median().add_prefix("median_"))


runs = load_runs(RUNS_DIR)
RUN_COLS = [
    "label", "model", "ok", "attempts", "seconds", "cost_usd", "arguments", "statements", "scenarios",
    "scenario_links", "edges", "isolated", "max_chain", "exclusions", "warnings",
]
failed = runs[~runs["ok"].fillna(False).astype(bool)]
print(f"{len(runs)} runs, {len(failed)} without a valid map, batch cost ${runs['cost_usd'].sum():.2f}")
if len(failed):
    display(failed.reindex(columns=["label", "model", "attempts", "errors", "issue_codes", "error"]))
runs.reindex(columns=RUN_COLS).sort_values(["label", "model"])

In [ ]:
v08_runs, v09_runs = load_runs(V08_DIR), load_runs(V09_DIR)
compare = pd.concat(
    {
        "v0.8": summarise(v08_runs[v08_runs["model"].isin(MODELS)]),
        "v0.9": summarise(v09_runs),
        "v0.10": summarise(runs),
    },
    names=["taxonomy"],
).round(2)
display(compare)
# Issue codes from the final attempt of each run, errors and warnings alike.
pd.DataFrame(runs.set_index("model")["issue_codes"].dropna().apply(pd.Series)).groupby(level=0).sum().T

### Observations

- 

## 2. Side-by-side model maps

`outputs/samples_v10/<doc>/models_list.mmd` shows haiku and opus as two columns in list view, including observations and scenarios.
- **Box colour** shows the claim type: RECOMMENDATION red, PREDICTION yellow, CAUSAL purple, VIEW blue, OBSERVATION grey, SUPPOSITION blue-grey.
- **The bracketed marker** shows the subkinds.
- **Scenario boxes** are blue-grey.

In [ ]:
for path in sorted(RUNS_DIR.glob("*/models_list.mmd")):
    print(path.relative_to(ROOT).as_posix())

## 3. Claim types and subkinds

In [ ]:
display(
    pd.crosstab(claims["model"], claims["claim_type"], margins=True)
    .reindex(index=[*MODELS, "All"], columns=[*TYPES, "All"], fill_value=0)
)
display(
    pd.crosstab(claims["model"], claims["claim_type"], normalize="index").round(2)
    .reindex(index=MODELS, columns=TYPES, fill_value=0)
)
sub_rows = claims.explode("subkind_list")
sub_rows["subkind_list"] = sub_rows["subkind_list"].fillna("(none)")
for claim_type in TYPES:
    sub = sub_rows[sub_rows["claim_type"] == claim_type]
    if len(sub):
        print(claim_type)
        display(pd.crosstab(sub["model"], sub["subkind_list"]).reindex(index=MODELS, fill_value=0))

In [ ]:
# PREDICTION subkind combinations, and one example per type x subkind (opus first).
display(claims[claims["claim_type"] == "PREDICTION"].groupby("model")["sub"].value_counts().unstack(0).fillna(0))
examples = (
    sub_rows.sort_values("model", key=lambda s: s.map({"opus": 0, "haiku": 1}))
    .groupby(["claim_type", "subkind_list"])
    .head(1)
)
examples.set_index(["claim_type", "subkind_list"]).sort_index()[["label", "model", "statement"]]

### Observations

- 

## 4. Hierarchy: arguments, links and roles

Per map: links between arguments (one argument's core claim reused in another argument's role), links from scenarios, arguments with no link, and recommendations whose `supporting` role holds another argument's core claim (that is, they are motivated).

In [ ]:
def hierarchy_stats(maps, version):
    rows = []
    for (doc, model), m in maps.items():
        types = {s.id: s.claim_type for s in m.statements}
        edges, sedges = argument_edges(m), scenario_edges(m)
        core_of = {a.core_claim: a.id for a in m.arguments}
        linked = {x for e in edges for x in (e[0], e[2])} | {e[2] for e in sedges}
        recs = [a for a in m.arguments if types[a.core_claim] == "RECOMMENDATION"]
        rows.append({
            "taxonomy": version,
            "label": LABEL[doc],
            "model": model,
            "arguments": len(m.arguments),
            "stmts_per_arg": round(len(m.statements) / len(m.arguments), 1),
            "arg_links": len(edges),
            "scenario_links": len(sedges),
            "isolated_args": sum(a.id not in linked for a in m.arguments),
            "recs": len(recs),
            "recs_motivated": sum(any(sid in core_of for sid in a.supporting) for a in recs),
            "core_types": dict(Counter(types[a.core_claim] for a in m.arguments)),
        })
    return pd.DataFrame(rows)


hierarchy = pd.concat([hierarchy_stats(maps09, "v0.9"), hierarchy_stats(maps10, "v0.10")], ignore_index=True)
display(hierarchy.drop(columns="core_types").pivot_table(
    index=["label", "model"], columns="taxonomy", values=["arguments", "arg_links", "isolated_args", "recs", "recs_motivated"],
    aggfunc="first",
).reindex(columns=["v0.9", "v0.10"], level=1))
hierarchy[hierarchy["taxonomy"] == "v0.10"][["label", "model", "core_types"]]

In [ ]:
# Role usage: statements by role and type, plus scenario references by role.
display(pd.crosstab(memberships["role"], memberships["claim_type"]).reindex(index=ROLES, columns=TYPES, fill_value=0))
scenario_refs = pd.DataFrame(
    [{"label": LABEL[doc], "model": model, "role": role} for (doc, model), m in maps10.items() for _, role, _ in scenario_edges(m)]
)
pd.crosstab(scenario_refs["role"], scenario_refs["model"]) if len(scenario_refs) else "no scenario references"

### Observations

- 

## 5. Coverage: must-keep claims

Two checks:
1. **Excluded sentences that look like must-keep claims.** These are the `MUST_KEEP_EXCLUDED` warnings: a regex probe over the excluded sentences.
2. **v0.9 statements with no match in the v0.10 map** of the same sample and model.

An unmatched statement is not necessarily missing. v0.10 synthesises core claims, so a v0.9 statement may have been folded into a longer v0.10 statement.

In [ ]:
rows = []
for (doc, model), m in maps10.items():
    issues_path = RUNS_DIR / doc / model / "issues.json"
    issues = json.loads(issues_path.read_text(encoding="utf-8")) if issues_path.exists() else []
    for issue in issues:
        if issue["code"] == "MUST_KEEP_EXCLUDED":
            index = int(re.search(r"\[(\d+)\]", issue["path"]).group(1))
            rows.append({"label": LABEL[doc], "model": model, "probe": issue["message"].split("like a ")[-1],
                         "reason": m.exclusions[index].reason, "text": m.exclusions[index].text})
must_keep = pd.DataFrame(rows, columns=["label", "model", "probe", "reason", "text"])
print(f"{len(must_keep)} excluded sentences look like must-keep claims")
display(pd.crosstab(must_keep["label"], must_keep["model"]) if len(must_keep) else None)
must_keep

In [ ]:
# Align each v0.9 map to the v0.10 map of the same sample and model (difflib, so cached).
alignment_path = DATA_DIR / "v09_alignment.csv"
if alignment_path.exists():
    align = pd.read_csv(alignment_path, encoding="utf-8-sig")
else:
    rows = []
    for (doc, model), m09 in maps09.items():
        m10 = maps10.get((doc, model))
        if m10 is None:
            continue
        new = {s.id: s for s in m10.statements}
        mapping = match_statements(m09, m10)
        for s in m09.statements:
            n = new.get(mapping.get(s.id))
            rows.append({
                "doc": doc, "model": model, "v09_id": s.id, "v09_type": s.claim_type, "v09_kind": s.proposition_kind,
                "v09_statement": s.statement, "v10_id": n.id if n else None, "v10_type": n.claim_type if n else None,
                "v10_sub": "+".join(n.subkinds) if n else None, "v10_statement": n.statement if n else None,
            })
    align = pd.DataFrame(rows)
    align.to_csv(alignment_path, index=False, encoding="utf-8-sig")
align["label"] = align["doc"].map(LABEL)
align["matched"] = align["v10_id"].notna()
is_must = align["v09_type"].isin(["RECOMMENDATION", "PREDICTION"]) | align["v09_kind"].isin(["valuation", "positioning"])
align["must_keep"] = is_must
display(align.groupby(["label", "model"])["matched"].agg(recall="mean", v09_statements="size").round(2))
must = align[align["must_keep"]]
display(must.groupby(["v09_type", "v09_kind"])["matched"].agg(recall="mean", n="size").round(2))
print("Unmatched v0.9 must-keep statements")
must[~must["matched"]][["label", "model", "v09_type", "v09_kind", "v09_statement"]]

### Observations

- 

## 6. Model invariance: haiku vs opus

Agreement on matched statements, compared with the v0.9 agreement on the same samples.

In [ ]:
pairwise_path = DATA_DIR / "pairwise.csv"
if pairwise_path.exists():
    pairwise = pd.read_csv(pairwise_path, encoding="utf-8-sig")
else:
    rows = []
    for doc in DOCS:
        if not all((doc, m) in maps10 for m in MODELS):
            continue
        a, b = maps10[(doc, "haiku")], maps10[(doc, "opus")]
        mapping = match_statements(a, b)
        sa, sb = {s.id: s for s in a.statements}, {s.id: s for s in b.statements}
        pairs_ = [(sa[x], sb[y]) for x, y in mapping.items()]
        n = len(pairs_)

        def share(test, pairs_=pairs_, n=n):
            return sum(test(x, y) for x, y in pairs_) / n if n else None

        rows.append({
            "doc": doc,
            "matched": n,
            "stmt_overlap_f1": 2 * n / (len(sa) + len(sb)),
            "type_agreement": share(lambda x, y: x.claim_type == y.claim_type),
            "subkind_agreement": share(lambda x, y: set(x.subkinds) == set(y.subkinds)),
            "joint_agreement": share(lambda x, y: x.claim_type == y.claim_type and set(x.subkinds) == set(y.subkinds)),
            "modality_agreement": share(lambda x, y: x.modality == y.modality),
        })
    pairwise = pd.DataFrame(rows)
    pairwise.to_csv(pairwise_path, index=False, encoding="utf-8-sig")
pairwise["label"] = pairwise["doc"].map(LABEL)
display(pairwise.set_index("label").drop(columns="doc").round(2))

v09_pairwise = pd.read_csv(V09_DIR / "claims_analysis" / "pairwise.csv", encoding="utf-8-sig")
v09_pairwise["label"] = v09_pairwise["doc"].map(LABEL)
pd.DataFrame({
    "v0.9 function (4)": v09_pairwise.set_index("label")["function_agreement"],
    "v0.10 type (6)": pairwise.set_index("label")["type_agreement"],
    "v0.9 kind (7)": v09_pairwise.set_index("label")["kind_agreement"],
    "v0.10 subkinds": pairwise.set_index("label")["subkind_agreement"],
    "v0.9 joint": v09_pairwise.set_index("label")["joint_agreement"],
    "v0.10 joint": pairwise.set_index("label")["joint_agreement"],
}).round(2)

In [ ]:
# Haiku statements aligned to the opus pivot: confusion on type, and the disagreements side by side.
aligned = claims.dropna(subset=["cluster_id"])
aligned = aligned[aligned["run_id"] != aligned["cluster_pivot"]].copy()
aligned["pivot_sub"] = aligned["cluster_pivot_subkinds"].fillna("")
aligned["type_ok"] = aligned["claim_type"] == aligned["cluster_pivot_claim_type"]
aligned["sub_ok"] = aligned["subkind_list"].map(set) == aligned["pivot_sub"].map(lambda v: set(v.split("|")) - {""})
print(f"{len(aligned)} haiku statements aligned to an opus statement")
heatmap(
    pd.crosstab(aligned["cluster_pivot_claim_type"], aligned["claim_type"]).reindex(index=TYPES, columns=TYPES, fill_value=0),
    "Claim type on aligned statements", "haiku", "opus",
)
plt.show()
pivot_rows = claims[claims["run_id"] == claims["cluster_pivot"]].set_index("cluster_id")


def side_by_side(mask):
    rows = aligned[mask]
    pivot = pivot_rows.loc[rows["cluster_id"]]
    return pd.DataFrame({
        "label": rows["label"].to_numpy(),
        "opus": (pivot["claim_type"] + " · " + pivot["sub"]).to_numpy(),
        "haiku": (rows["claim_type"] + " · " + rows["sub"]).to_numpy(),
        "opus statement": pivot["statement"].to_numpy(),
        "haiku statement": rows["statement"].to_numpy(),
    })


print("Type disagreements")
display(side_by_side(~aligned["type_ok"]))
print("Same type, different subkinds")
side_by_side(aligned["type_ok"] & ~aligned["sub_ok"])

### Observations

- 

## 7. Where did the v0.9 types go?

In [ ]:
matched = align[align["matched"]].copy()
print(f"{len(matched)}/{len(align)} v0.9 statements matched a v0.10 statement")
display(
    pd.crosstab(matched["v09_type"], matched["v10_type"], margins=True)
    .reindex(index=["RECOMMENDATION", "PREDICTION", "ASSESSMENT", "OBSERVATION", "All"], columns=[*TYPES, "All"], fill_value=0)
)
# The three most common v0.10 destinations of each v0.9 function x kind.
matched["v09"] = matched["v09_type"] + " · " + matched["v09_kind"].fillna("?")
matched["v10"] = matched["v10_type"] + " · " + matched["v10_sub"].fillna("")
dest = matched.groupby("v09")["v10"].value_counts(normalize=True).round(2).groupby(level=0).head(3).to_frame("share")
dest.join(matched.groupby("v09").size().rename("n"), on="v09").query("n >= 5")

### Observations

- 

## 8. Suppositions, conditions and scenarios

In [ ]:
sup = claims[claims["claim_type"] == "SUPPOSITION"].copy()
sup["in_conditions"] = sup["role_set"].fillna("").str.contains("conditions")
sup["in_scenario"] = sup["scenario_slots"].fillna("").str.contains("assumptions")
print(f"{len(sup)} SUPPOSITION statements")
display(sup.groupby("model")[["in_conditions", "in_scenario"]].sum())
display(sup[SHOW + ["role_set", "scenario_slots"]])

# Conditional wording outside suppositions: candidates for an unsplit premise.
CONDITIONAL = r"\b(?:if|unless|should the|in the event|assuming|after a|in a \w+ scenario)\b"
cond = claims[(claims["claim_type"] != "SUPPOSITION") & probe(claims, {"c": CONDITIONAL})["c"]]
print(f"{len(cond)} non-supposition statements still contain a conditional clause")
display(pd.crosstab(cond["claim_type"], cond["model"]))
cond[SHOW + ["role_set", "scenario_slots"]]

In [ ]:
rows = []
for (doc, model), m in maps10.items():
    types = {s.id: s.claim_type for s in m.statements}
    text = {s.id: s.statement for s in m.statements}
    refs = {}
    for scenario_id, role, argument_id in scenario_edges(m):
        refs.setdefault(scenario_id, []).append(f"{argument_id}.{role}")
    for scenario in m.scenarios:
        outcomes = [text[s] for s in scenario.outcomes]
        rows.append({
            "label": LABEL[doc],
            "model": model,
            "id": scenario.id,
            "scenario": scenario.label,
            "referenced_from": ", ".join(refs.get(scenario.id, [])) or "-",
            "assumptions": ", ".join(types.get(s, "?") for s in scenario.assumptions),
            "outcomes": ", ".join(types.get(s, "?") for s in scenario.outcomes),
            "outcomes_restating_premise": sum(bool(re.search(CONDITIONAL, o, re.IGNORECASE)) for o in outcomes),
            "recommendations": len(scenario.recommendations),
        })
scenarios = pd.DataFrame(rows)
scenarios

### Observations

- 

## 9. CAUSAL claims and the function boundaries

In [ ]:
BOUNDARY_PROBES = {
    "causal_wording": (
        r"\b(?:drive[sn]?|driven|drove|support\w*|weigh\w*|because|due to|lead\w* to|push\w*|reflect\w*|caus\w*"
        r"|boost\w*|hurt\w*|explain\w*|on the back of|as a result|amid|depend\w*|requir\w*|vulnerab\w*)\b"
    ),
    "future_marker": r"\b(?:will|would|expect\w*|forecast\w*|by (?:end|year-end|mid)|next (?:year|week|month|quarter))\b",
    "number": r"\d",
}
display(probe(claims, BOUNDARY_PROBES).groupby(claims["claim_type"]).mean().round(2).T.reindex(columns=TYPES))
display(pd.crosstab(claims["claim_type"], claims["modality"], normalize="index").round(2).reindex(TYPES))
causal = claims[claims["claim_type"] == "CAUSAL"]
is_future = probe(causal, {"f": BOUNDARY_PROBES["future_marker"]})["f"]
print(f"{len(causal)} CAUSAL statements; {is_future.sum()} with future wording (candidate predictions)")
causal[is_future][SHOW + ["role_set"]]

In [ ]:
# CAUSAL samples per model, and recommendations with their subkind and attribution.
display(causal.groupby("model").head(6)[SHOW + ["role_set"]])
claims[claims["claim_type"] == "RECOMMENDATION"][SHOW + ["role_set"]]

### Observations

- 

## 10. Attribution

In [ ]:
attributed = claims.dropna(subset=["attribution"])
print(f"{len(attributed)} of {len(claims)} statements attributed to someone other than the author")
display(pd.crosstab(attributed["claim_type"], attributed["model"]).reindex(index=TYPES, columns=MODELS, fill_value=0))
display(attributed["attribution"].value_counts().head(20).to_frame("statements"))
attributed[SHOW].head(30)

### Observations

- 

## 11. Summary

- 